# Exercise 09 — Question Answering

In notebook 0 you saw how hard it is to score a generated text when many answers are correct. **Question answering** (QA) is a task where the answer is usually short and checkable, which makes it one of the most common ways to evaluate language models, and one of their most common uses.

There are two ways to build a QA system with what you know so far:

- **Extractive QA**: the answer is a **span** of a given passage. A BERT-style encoder reads the question and the passage together and predicts where the answer starts and where it ends. Such a system can only answer with words from the passage, but it can never make something up.
- **Generative QA**: a decoder (or encoder–decoder) model **writes** the answer, with or without a passage to read from. It can answer any kind of question, in any words, but it can also produce fluent, confident answers that are wrong: **hallucinations**.

## What you will do
1. **SQuAD**: the standard extractive QA dataset, and how its answers are stored.
2. **Extractive QA by hand**: start and end logits, and choosing the best valid span.
3. **Evaluating QA**: exact match and F1, implemented as in the official SQuAD script.
4. **Questions without an answer**: SQuAD 2.0, the "no answer" score, and a threshold.
5. **Generative QA**: a small chat model, with and without the passage, and why exact match is unfair to it.
6. **Multiple-choice questions**: to check your understanding.

## How to work through it
- Run the cells **in order** and fill in **every `# TODO`**.
- Tasks are numbered (**1.1**, **1.2**, …). Tasks that say *Your answer here* want a short written answer, not code.
- Most implementation tasks are followed by a **✅ Check** cell that verifies your work automatically. Run it and make sure it passes before you move on.

> **Heads up: this notebook downloads two models**, `deepset/bert-base-cased-squad2` (~430 MB) and `Qwen/Qwen2.5-0.5B-Instruct` (~1 GB), and the SQuAD and SQuAD 2.0 validation sets (~20 MB). Nothing is trained, and everything runs on a CPU. The evaluations in Parts 3 and 4 take about a minute each, and Part 5 takes a few minutes.

In [ ]:
%matplotlib inline
import re
import string
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoModelForQuestionAnswering, AutoTokenizer

torch.manual_seed(0)

# Use the GPU if there is one; a CPU is enough for this notebook.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device} device")

## 1. SQuAD

The **Stanford Question Answering Dataset** ([SQuAD](https://huggingface.co/datasets/rajpurkar/squad); [Rajpurkar et al., 2016](https://arxiv.org/abs/1606.05250)) contains more than 100,000 questions that crowdworkers wrote about paragraphs from Wikipedia articles. The answer to every question is a span of its paragraph (the **context**). For the validation set, several workers answered every question, so there are up to five reference answers, often the same span or a slightly longer or shorter one.

We only need the validation set (10,570 questions).

In [ ]:
squad = load_dataset("rajpurkar/squad", split="validation")
print(squad)

example = squad[0]
print("\nCONTEXT:\n" + example["context"])
print("\nQUESTION: " + example["question"])
print("ANSWERS: ", example["answers"])

An answer is stored as its `text` and its `answer_start`, the **character** position in the context where it begins. Character positions are what the original annotation tool recorded; they are independent of any tokenizer, which is why datasets store them rather than token positions.

**1.1 Complete `answer_spans`**, which returns the `(start, end)` character positions of all reference answers of an example, so that `context[start:end]` is the answer text.

In [ ]:
def answer_spans(example):
    # A list of (start, end) character positions of the reference answers
    return ...  # TODO


for start, end in answer_spans(example):
    print((start, end), repr(example["context"][start:end]))

In [ ]:
# ✅ Check answer_spans
for ex in squad.select(range(200)):
    spans = answer_spans(ex)
    assert len(spans) == len(ex["answers"]["text"]), "return one span per reference answer"
    assert [ex["context"][s:e] for s, e in spans] == ex["answers"]["text"], "context[start:end] should be the answer text"
print("Looks good ✅")

## 2. Extractive QA by hand

An extractive QA model is a BERT-style encoder with a tiny head on top: a linear layer that maps the final hidden state of every token to **two numbers**, a *start logit* and an *end logit*. The question and the context are fed in as one sequence,

`[CLS] question [SEP] context [SEP]`,

so that every context token can attend to the question. During fine-tuning, the model learns to give the highest start logit to the first token of the answer and the highest end logit to its last token. A softmax over the positions turns them into a distribution over where the answer starts (and ends).

We use [`deepset/bert-base-cased-squad2`](https://huggingface.co/deepset/bert-base-cased-squad2), a BERT-base model that was fine-tuned on SQuAD 2.0 (Part 4).

In [ ]:
QA_NAME = "deepset/bert-base-cased-squad2"

qa_tokenizer = AutoTokenizer.from_pretrained(QA_NAME)
qa_model = AutoModelForQuestionAnswering.from_pretrained(QA_NAME).to(device)
qa_model.eval()
print(f"{sum(p.numel() for p in qa_model.parameters()):,} parameters")
print(qa_model.qa_outputs)  # the QA head: hidden size -> (start logit, end logit)

The function below tokenizes a question and its context and runs the model. Three details:

- `truncation="only_second"` cuts the *context* (never the question) when the pair is longer than `max_length` tokens. A real system splits long contexts into several overlapping windows; SQuAD paragraphs almost always fit.
- `return_offsets_mapping=True` returns, for every token, the `(start, end)` **character** positions it came from. This lets us map a predicted span of tokens back to text in the context.
- `sequence_ids()` tells, for every token, whether it belongs to the question (`0`), the context (`1`), or is a special token (`None`).

In [ ]:
def qa_forward(question, context, max_length=384):
    # Start and end logits for every token, the character offsets, and a mask of the context tokens
    inputs = qa_tokenizer(
        question, context, truncation="only_second", max_length=max_length, return_offsets_mapping=True, return_tensors="pt"
    )
    offsets = inputs.pop("offset_mapping")[0].tolist()
    context_mask = torch.tensor([s == 1 for s in inputs.sequence_ids(0)])
    with torch.no_grad():
        out = qa_model(**inputs.to(device))
    return out.start_logits[0].cpu(), out.end_logits[0].cpu(), offsets, context_mask


question, context = example["question"], example["context"]
start_logits, end_logits, offsets, context_mask = qa_forward(question, context)
tokens = qa_tokenizer.convert_ids_to_tokens(qa_tokenizer(question, context)["input_ids"])
print(f"{len(tokens)} tokens, {int(context_mask.sum())} of them in the context")
for i in torch.topk(start_logits, 3).indices.tolist():
    print(f"start candidate {i:3} {tokens[i]!r:12} logit {start_logits[i]:.2f}")
for i in torch.topk(end_logits, 3).indices.tolist():
    print(f"end candidate   {i:3} {tokens[i]!r:12} logit {end_logits[i]:.2f}")

### Choosing the span

The simplest decoding takes the argmax of the start logits and the argmax of the end logits independently. That usually works, but nothing guarantees a **valid** span: the end could come before the start, the span could lie in the question, or it could be absurdly long.

A span from token $s$ to token $e$ gets the score $\text{start\_logit}_s + \text{end\_logit}_e$ (the log of the product of the two probabilities, up to a constant). The best answer is the span with the highest score among the valid ones:

- $s$ and $e$ are both tokens of the **context**,
- $s \le e$, and the span has at most `max_answer_len` tokens.

**2.1 Complete `best_span`.** Compute the scores of all pairs at once as a matrix, set the scores of invalid pairs to $-\infty$, and take the argmax.

Hints:
- `start_logits[:, None] + end_logits[None, :]` is the matrix with entry `[s, e]` = start logit of `s` + end logit of `e`.
- `torch.ones(n, n, dtype=torch.bool).triu()` is `True` where `s <= e`, and `.triu().tril(k)` additionally requires `e <= s + k`.
- `matrix.argmax()` gives the index into the flattened matrix; `divmod(index, n)` turns it into a row and a column.

In [ ]:
def best_span(start_logits, end_logits, context_mask, max_answer_len=30):
    # (start token, end token, score) of the best valid span; end is inclusive
    n = len(start_logits)
    scores = ...  # TODO: (n, n) matrix of start_logits[s] + end_logits[e]
    valid = ...   # TODO: (n, n) boolean matrix: s <= e < s + max_answer_len, and both s and e in the context
    scores = scores.masked_fill(~valid, float("-inf"))
    s, e = ...    # TODO: the row and column of the highest score
    return s, e, scores[s, e].item()


s, e, score = best_span(start_logits, end_logits, context_mask)
print(s, e, score, tokens[s : e + 1])

In [ ]:
# ✅ Check best_span against a brute-force search
def brute_force(start_logits, end_logits, context_mask, max_answer_len=30):
    best = (None, None, float("-inf"))
    for s in range(len(start_logits)):
        for e in range(s, min(s + max_answer_len, len(end_logits))):
            if context_mask[s] and context_mask[e] and start_logits[s] + end_logits[e] > best[2]:
                best = (s, e, (start_logits[s] + end_logits[e]).item())
    return best


g = torch.Generator().manual_seed(0)
for trial in range(20):
    n = 40
    sl, el = torch.randn(n, generator=g), torch.randn(n, generator=g)
    mask = torch.zeros(n, dtype=torch.bool)
    mask[10:38] = True
    mine, theirs = best_span(sl, el, mask, max_answer_len=5), brute_force(sl, el, mask, max_answer_len=5)
    assert mine[:2] == theirs[:2] and abs(mine[2] - theirs[2]) < 1e-5, f"best_span returned {mine}, the best valid span is {theirs}"
print("Looks good ✅")

**2.2 Complete `answer_question`**, which returns the answer as text, together with its score. Use the character offsets of the first and the last token of the span to cut the answer out of the **context**. (Decoding the tokens with the tokenizer would also work, but cutting it out of the context gives back exactly the original characters, including spaces and capitalisation.)

In [ ]:
def answer_question(question, context, max_answer_len=30):
    # The best answer span (as a string) and its score
    start_logits, end_logits, offsets, context_mask = qa_forward(question, context)
    s, e, score = best_span(start_logits, end_logits, context_mask, max_answer_len)
    answer = ...  # TODO: the characters of the context from the start of token s to the end of token e
    return answer, score


for ex in squad.select(range(0, 50, 10)):
    answer, score = answer_question(ex["question"], ex["context"])
    print(f"Q: {ex['question']}\n   predicted: {answer!r} (score {score:.1f})   references: {ex['answers']['text']}\n")

toy_context = "Copenhagen is the capital of Denmark. Aarhus, the second largest city, is home to Aarhus University, founded in 1928."
print(answer_question("What is the capital of Denmark?", toy_context))

In [ ]:
# ✅ Check answer_question
answer, score = answer_question("What is the capital of Denmark?", toy_context)
assert isinstance(answer, str) and isinstance(score, float), "return the answer as a string and the score as a float"
assert answer == "Copenhagen", f"expected 'Copenhagen', got {answer!r}"
assert answer_question("When was Aarhus University founded?", toy_context)[0] == "1928"
answer, _ = answer_question(squad[0]["question"], squad[0]["context"])
assert answer in squad[0]["context"], "the answer should be cut out of the context"
print("Looks good ✅")

**2.3 What can an extractive model not do?** Think of the context *"Marie Curie was awarded the Nobel Prize in Physics in 1903 and the Nobel Prize in Chemistry in 1911."* and the questions *"Did Marie Curie win a Nobel Prize?"* and *"How many Nobel Prizes did Marie Curie win?"*. What would an extractive model answer? Name two more kinds of questions that it cannot answer correctly. Try them with `answer_question` if you like.

---

*Your answer here:*

---

## 3. Evaluating QA

QA answers are short, so they can be compared with the references directly. The official SQuAD evaluation uses two metrics:

- **Exact match (EM)**: 1 if the prediction is the same as one of the references, 0 otherwise.
- **F1**: the token-level F1 between the prediction and a reference: the overlap of the two bags of words, with precision divided by the length of the prediction and recall by the length of the reference (just like ROUGE-1 in notebook 0). The best F1 over the references counts.

Both compare **normalised** answers: lower-cased, without punctuation, without the articles *a*, *an* and *the*, and with all whitespace collapsed to single spaces. So *"The Denver Broncos!"* and *"Denver Broncos"* are an exact match.

**3.1 Complete `normalize_answer`, `exact_match` and `f1_score`** as in the [official evaluation script](https://rajpurkar.github.io/SQuAD-explorer/).

Hints:
- `string.punctuation` is a string of all punctuation characters.
- `re.sub(r"\b(a|an|the)\b", " ", text)` removes the articles as whole words.
- `Counter(a) & Counter(b)` keeps the minimum count of every token, so `sum((Counter(a) & Counter(b)).values())` is the number of common tokens.
- If the prediction or the reference is empty after normalising (which happens for unanswerable questions in Part 4), F1 is 1 if both are empty and 0 otherwise.

In [ ]:
def normalize_answer(text):
    # Lower-case, remove punctuation, remove articles, and collapse whitespace
    text = ...  # TODO
    return text


def exact_match(prediction, references):
    # 1.0 if the normalised prediction equals one of the normalised references, else 0.0
    return ...  # TODO


def f1_single(prediction, reference):
    pred_tokens, ref_tokens = normalize_answer(prediction).split(), normalize_answer(reference).split()
    # TODO: the token-level F1 (handle empty answers first)
    ...


def f1_score(prediction, references):
    # The best token-level F1 over the references
    return max(f1_single(prediction, reference) for reference in references)


print(normalize_answer("  The Denver Broncos!  "))
print(exact_match("The Denver Broncos!", ["Denver Broncos", "the Broncos"]), f1_score("the Broncos of Denver", ["Denver Broncos"]))

In [ ]:
# ✅ Check the metrics
assert normalize_answer("  The Denver Broncos!  ") == "denver broncos"
assert normalize_answer("A theory, an idea") == "theory idea", "remove the articles a and an as whole words (but not 'the' inside 'theory')"
assert exact_match("The Denver Broncos!", ["Denver Broncos"]) == 1.0 and exact_match("Broncos", ["Denver Broncos"]) == 0.0
assert isinstance(exact_match("x", ["x"]), float), "exact_match should return a float"
assert abs(f1_score("the Broncos of Denver", ["Denver Broncos"]) - 0.8) < 1e-9, "precision 2/3, recall 1, F1 0.8"
assert f1_score("Broncos", ["Denver Broncos", "Broncos"]) == 1.0, "take the best reference"
assert f1_score("", [""]) == 1.0 and f1_score("Broncos", [""]) == 0.0 and f1_score("", ["Broncos"]) == 0.0, "handle empty answers"
print("Looks good ✅")

**3.2 Evaluate the extractive model** on the first `N_EVAL = 200` validation questions: store every prediction, and print the mean EM and F1 (in percent, as they are usually reported).

In [ ]:
N_EVAL = 200
eval_set = squad.select(range(N_EVAL))

predictions = []
for ex in eval_set:
    ...  # TODO: append the predicted answer for ex

em = ...  # TODO: the mean exact match over the eval set, in percent
f1 = ...  # TODO: the mean F1, in percent
print(f"EM {em:.1f}   F1 {f1:.1f}")

In [ ]:
# ✅ Check your evaluation
assert len(predictions) == N_EVAL and all(isinstance(p, str) for p in predictions), "store one prediction (a string) per question"
assert 50 < em <= 100 and em <= f1 <= 100, "EM and F1 should be in percent, with EM <= F1"
print("Looks good ✅")

The cell below prints the questions on which the model got **no exact match** but partial credit from F1.

In [ ]:
for p, ex in zip(predictions, eval_set):
    if exact_match(p, ex["answers"]["text"]) == 0 and f1_score(p, ex["answers"]["text"]) > 0:
        print(f"Q: {ex['question']}\n   predicted: {p!r}   references: {sorted(set(ex['answers']['text']))}   F1 {f1_score(p, ex['answers']['text']):.2f}\n")

**3.3 Look at the partial matches.** Are these predictions wrong? What does this tell you about the gap between EM and F1, and about what EM measures? Why do SQuAD's validation questions have several reference answers?

---

*Your answer here:*

---

## 4. Questions without an answer

In SQuAD every question has an answer in its paragraph, so a model can always return *some* span. In practice, the passage often does not contain the answer, and a good system should say so instead of returning the most plausible span. **SQuAD 2.0** ([Rajpurkar et al., 2018](https://arxiv.org/abs/1806.03822)) adds about 50,000 **unanswerable** questions, written by crowdworkers to look similar to answerable ones (they mention the right entities, but ask for something the paragraph does not say). Their list of reference answers is empty.

Models trained on SQuAD 2.0 learn to point at the `[CLS]` token (position 0) when there is no answer. The **no-answer score** is `start_logits[0] + end_logits[0]`. The model abstains if the no-answer score is higher than the score of the best span by more than a **threshold**, which you can tune on validation data (a higher threshold makes the model answer more often).

In [ ]:
squad_v2 = load_dataset("rajpurkar/squad_v2", split="validation").shuffle(seed=0).select(range(200))
n_unanswerable = sum(len(ex["answers"]["text"]) == 0 for ex in squad_v2)
print(f"{len(squad_v2)} questions, {n_unanswerable} of them unanswerable")

ex = next(ex for ex in squad_v2 if len(ex["answers"]["text"]) == 0)
print("\nCONTEXT:\n" + ex["context"])
print("\nUNANSWERABLE QUESTION: " + ex["question"])
print("best span anyway:", answer_question(ex["question"], ex["context"]))

**4.1 Complete `answer_or_abstain`**, which returns the empty string `""` when the model abstains. Returning `""` makes our metrics work unchanged: for an unanswerable question we use `[""]` as the list of references, and `f1_single` gives 1 only if the prediction is empty as well.

In [ ]:
def answer_or_abstain(question, context, threshold=0.0, max_answer_len=30):
    # The best span, or "" if the no-answer score beats the best span's score by more than the threshold
    start_logits, end_logits, offsets, context_mask = qa_forward(question, context)
    s, e, span_score = best_span(start_logits, end_logits, context_mask, max_answer_len)
    null_score = ...  # TODO
    if ...:  # TODO
        return ""
    return context[offsets[s][0] : offsets[e][1]]


print(repr(answer_or_abstain("What is the capital of Denmark?", toy_context)))
print(repr(answer_or_abstain("What is the capital of Sweden?", toy_context)))

In [ ]:
# ✅ Check answer_or_abstain
assert answer_or_abstain("What is the capital of Denmark?", toy_context) == "Copenhagen"
assert answer_or_abstain("Who won the football World Cup in 2010?", toy_context) == "", "the context says nothing about football"
assert answer_or_abstain("Who won the football World Cup in 2010?", toy_context, threshold=1e9) != "", "with a huge threshold the model should always answer"
assert answer_or_abstain("What is the capital of Denmark?", toy_context, threshold=-1e9) == "", "with a hugely negative threshold the model should always abstain"
print("Looks good ✅")

### Tuning the threshold

Running the model is the slow part, but the threshold only matters *after* it. The cell below runs the model once per question and stores the difference `null_score - span_score` together with the best span.

**4.2 Complete the threshold sweep**: for every threshold, compute the F1 on the answerable questions, on the unanswerable ones, and on all of them.

In [ ]:
records = []
for ex in squad_v2:
    start_logits, end_logits, offsets, context_mask = qa_forward(ex["question"], ex["context"])
    s, e, span_score = best_span(start_logits, end_logits, context_mask)
    records.append({
        "margin": (start_logits[0] + end_logits[0]).item() - span_score,
        "span": ex["context"][offsets[s][0] : offsets[e][1]],
        "references": ex["answers"]["text"] or [""],
        "answerable": len(ex["answers"]["text"]) > 0,
    })

thresholds = np.linspace(-10, 10, 41)
f1_answerable, f1_unanswerable, f1_all = [], [], []
for threshold in thresholds:
    preds = ...  # TODO: the prediction for every record at this threshold ("" or the span)
    f1s = np.array([f1_score(p, r["references"]) for p, r in zip(preds, records)])
    answerable = np.array([r["answerable"] for r in records])
    f1_answerable.append(...)    # TODO: mean F1 on the answerable questions
    f1_unanswerable.append(...)  # TODO: mean F1 on the unanswerable questions
    f1_all.append(...)           # TODO: mean F1 on all questions

plt.figure(figsize=(7, 4))
plt.plot(thresholds, f1_answerable, label="answerable")
plt.plot(thresholds, f1_unanswerable, label="unanswerable")
plt.plot(thresholds, f1_all, label="all", linewidth=3)
plt.xlabel("threshold (higher = answer more often)")
plt.ylabel("F1")
plt.legend()
plt.show()
best = thresholds[int(np.argmax(f1_all))]
print(f"best threshold {best:.1f}: F1 {100 * max(f1_all):.1f} (at 0.0: {100 * f1_all[len(thresholds) // 2]:.1f})")

In [ ]:
# ✅ Check the sweep
assert len(f1_all) == len(thresholds) == len(f1_answerable) == len(f1_unanswerable), "one value per threshold"
assert all(np.diff(f1_unanswerable) <= 1e-9), "a higher threshold means abstaining less, so F1 on unanswerable questions can only go down"
assert all(np.diff(f1_answerable) >= -1e-9), "a higher threshold means answering more, so F1 on answerable questions can only go up"
n_answerable = sum(r["answerable"] for r in records)
assert np.allclose(f1_all, (n_answerable * np.array(f1_answerable) + (len(records) - n_answerable) * np.array(f1_unanswerable)) / len(records)), "f1_all should be the mean over all questions"
print("Looks good ✅")

**4.3 The trade-off.** Describe the three curves. Why is it a bad idea to report the F1 on answerable and unanswerable questions *separately* without saying which threshold was used? In which applications would you choose a threshold that abstains more often than the best one, and in which one that abstains less?

---

*Your answer here:*

---

## 5. Generative QA

A generative model writes its answer token by token, so it is not limited to spans. We use [`Qwen/Qwen2.5-0.5B-Instruct`](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct), the chat version of the base model you fine-tuned in week 7. It is tiny for a chat model, but runs on a CPU.

We compare two settings:

- **Open-book**: the model gets the passage and the question, like the extractive model.
- **Closed-book**: the model only gets the question and has to answer from what it memorised during pretraining (its *parametric knowledge*).

In [ ]:
GEN_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

gen_tokenizer = AutoTokenizer.from_pretrained(GEN_NAME)
gen_model = AutoModelForCausalLM.from_pretrained(GEN_NAME).to(device)
gen_model.eval()
print(f"{sum(p.numel() for p in gen_model.parameters()):,} parameters")

**5.1 Complete `generate_answer`.** Build a conversation with a system message that asks for a **short** answer (a few words, not a sentence), and a user message with the question and, if one is given, the context. Then apply the chat template (week 7), generate greedily (last week) and return only the new text.

In [ ]:
def generate_answer(question, context=None, max_new_tokens=32):
    # A short answer generated by the chat model, with or without a context
    if context is None:
        user = ...  # TODO: the question only
    else:
        user = ...  # TODO: the context and the question
    messages = [
        {"role": "system", "content": ...},  # TODO: ask for a short answer of a few words
        {"role": "user", "content": user},
    ]
    inputs = gen_tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(device)
    with torch.no_grad():
        outputs = ...  # TODO: greedy generation of at most max_new_tokens new tokens
    new_ids = ...  # TODO: the new tokens only
    return gen_tokenizer.decode(new_ids, skip_special_tokens=True).strip()


print(repr(generate_answer("What is the capital of Denmark?", toy_context)))
print(repr(generate_answer("Did Marie Curie win a Nobel Prize?", "Marie Curie was awarded the Nobel Prize in Physics in 1903 and the Nobel Prize in Chemistry in 1911.")))
print(repr(generate_answer("How many Nobel Prizes did Marie Curie win?", "Marie Curie was awarded the Nobel Prize in Physics in 1903 and the Nobel Prize in Chemistry in 1911.")))

In [ ]:
# ✅ Check generate_answer
answer = generate_answer("What is the capital of Denmark?", toy_context)
assert isinstance(answer, str) and "Copenhagen" in answer, f"expected an answer containing 'Copenhagen', got {answer!r}"
assert "Context" not in answer and "system" not in answer, "return only the new text, without the prompt"
assert len(generate_answer("What is the capital of France?", max_new_tokens=5).split()) <= 6, "respect max_new_tokens"
print("Looks good ✅")

### Comparing the systems

Exact match is harsh on a generative model: *"The Denver Broncos won."* is a correct answer but gets an EM of 0 against *"Denver Broncos"*. A common, more lenient metric for generated answers is **answer containment** (sometimes called *accuracy* or *match*): 1 if the normalised reference appears inside the normalised prediction.

**5.2 Complete `contains_answer`**, and run the comparison of the extractive model and the generative model, open-book and closed-book, on the first `N_GEN = 50` questions of `eval_set`.

In [ ]:
def contains_answer(prediction, references):
    # 1.0 if one of the normalised references occurs in the normalised prediction, else 0.0
    return ...  # TODO


N_GEN = 50
gen_set = eval_set.select(range(N_GEN))
systems = {
    "extractive": predictions[:N_GEN],
    "generative, open-book": [generate_answer(ex["question"], ex["context"]) for ex in gen_set],
    "generative, closed-book": [generate_answer(ex["question"]) for ex in gen_set],
}

print(f"{'':25}   EM     F1  contains  words")
for name, preds in systems.items():
    refs = [ex["answers"]["text"] for ex in gen_set]
    em = 100 * np.mean([exact_match(p, r) for p, r in zip(preds, refs)])
    f1 = 100 * np.mean([f1_score(p, r) for p, r in zip(preds, refs)])
    contains = 100 * np.mean([contains_answer(p, r) for p, r in zip(preds, refs)])
    words = np.mean([len(p.split()) for p in preds])
    print(f"{name:25} {em:5.1f}  {f1:5.1f}  {contains:8.1f}  {words:5.1f}")

In [ ]:
# ✅ Check contains_answer
assert contains_answer("The Denver Broncos won the game.", ["Denver Broncos"]) == 1.0
assert contains_answer("Broncos", ["Denver Broncos"]) == 0.0, "the whole reference has to be in the prediction"
assert contains_answer("In 1928.", ["1928", "in 1928"]) == 1.0
assert isinstance(contains_answer("a", ["a"]), float), "return a float"
print("Looks good ✅")

In [ ]:
for ex, open_book, closed_book in list(zip(gen_set, systems["generative, open-book"], systems["generative, closed-book"]))[:12]:
    print(f"Q: {ex['question']}\n   reference: {ex['answers']['text'][0]!r}\n   open-book: {open_book!r}\n   closed-book: {closed_book!r}\n")

**5.3 Compare the three systems.**
1. How do the generative model's EM, F1 and containment compare? What does the difference tell you about which metric to use for a generative system?
2. Look at the closed-book answers. Are the wrong ones marked as uncertain, or do they sound as confident as the right ones? Why is this especially dangerous? Why does the open-book setting help so much?
3. A loophole of the containment metric: what kind of answer would get a high containment score without being a good answer?

---

*Your answer here:*

---

## 6. MCQ

Answer each question by writing the letter of your choice (A–D) after **Answer:**.

---

### 6.1. Extractive QA

What does the QA head of an extractive model like `bert-base-cased-squad2` predict?

A. The answer text, token by token<br>
B. For every token, a start logit and an end logit, from which the answer span is chosen<br>
C. One logit per word in the vocabulary, for the first word of the answer<br>
D. A yes/no decision for every sentence of the context<br>

**Answer:** 

---

### 6.2. Valid spans

Why can taking `argmax(start_logits)` and `argmax(end_logits)` independently go wrong?

A. The argmax is not differentiable<br>
B. The two positions may not form a valid span: the end may come before the start, or lie in the question<br>
C. The logits must first be turned into probabilities with a softmax<br>
D. The argmax always returns the `[CLS]` token<br>

**Answer:** 

---

### 6.3. Exact match and F1

The references are `["Denver Broncos"]` and the prediction is `"the Broncos"`. What are EM and F1 after the SQuAD normalisation?

A. EM 0, F1 0<br>
B. EM 0, F1 0.67<br>
C. EM 1, F1 1<br>
D. EM 0, F1 0.5<br>

**Answer:** 

---

### 6.4. Unanswerable questions

How does a model fine-tuned on SQuAD 2.0 indicate that a question has no answer in the context?

A. It returns an empty list of logits<br>
B. It gives the highest start and end scores to the `[CLS]` token, and abstains if that score beats the best span by more than a threshold<br>
C. It generates the string "unanswerable"<br>
D. It returns the whole context as the answer<br>

**Answer:** 

---

### 6.5. Closed-book QA

A chat model answers a factual question **without** any context and gets it wrong in a fluent, confident sentence. What is this called, and what is the most direct fix among the options?

A. Overfitting; train the model for fewer epochs<br>
B. A hallucination; give the model relevant passages to answer from (retrieval-augmented generation)<br>
C. Exposure bias; use beam search instead of greedy decoding<br>
D. A tokenization error; use a larger vocabulary<br>

**Answer:** 